# Comparing and interrogating model performance

In [21]:
import json
import pandas as pd
import joblib
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

data_path = "../data"

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

In [22]:
import pandas as pd

# Ground truth from the fixed test set
comparison_df = pd.DataFrame({
    "Date": [quiz["date"] for quiz in test_set],
    "True answer": [quiz["answer"] for quiz in test_set]
})

# Standardised prediction files
model_files = {
    "TF-IDF + LR": "../data/predictions_tfidf_lr.csv",
    "MiniLM + LR": "../data/predictions_minilm_lr.csv",
    "MiniLM +  MLP": "../data/predictions_minilm_mlp.csv",
    "DistilBERT Headline": "../data/predictions_distilbert_headline.csv",
    "Distilbert Concat": "../data/predictions_distilbert_concat.csv",
    "Distilbert Shared": "../data/predictions_distilbert_shared.csv",
    "Llama QLoRA": "../data/predictions_llama_qlora.csv",
    "Claude Haiku 4.5": "../data/predictions_claude_haiku.csv",
    "GPT-5 Nano": "../data/predictions_gpt5_nano.csv",
    "Claude Fable 5.1": "../data/predictions_claude_fable.csv",
    "GPT-6 Astra": "../data/predictions_gpt6_astra.csv",
}

# Merge each model's results into the comparison dataframe
for model_name, filepath in model_files.items():
    model_df = pd.read_csv(filepath)

    model_df = model_df.rename(columns={
        "date": "Date",
        "prediction": f"{model_name} prediction",
        "confidence": f"{model_name} confidence",
    })

    comparison_df = comparison_df.merge(
        model_df,
        on="Date",
        how="left",
        validate="one_to_one",
    )

# Sort chronologically
comparison_df = comparison_df.sort_values("Date").reset_index(drop=True)

comparison_df

,Date,True answer,TF-IDF + LR prediction,TF-IDF + LR confidence,MiniLM + LR prediction,MiniLM + LR confidence,MiniLM + MLP prediction,MiniLM + MLP confidence,DistilBERT Headline prediction,DistilBERT Headline confidence,...,Llama QLoRA prediction,Llama QLoRA confidence,Claude Haiku 4.5 prediction,Claude Haiku 4.5 confidence,GPT-5 Nano prediction,GPT-5 Nano confidence,Claude Fable 5.1 prediction,Claude Fable 5.1 confidence,GPT-6 Astra prediction,GPT-6 Astra confidence
0,2019-11-23,0,3,0.338882,0,0.550084,0,0.405892,0,0.153801,...,1,0.648438,1,NaN,1,NaN,0,NaN,0,NaN
1,2020-01-18,3,3,0.280240,3,0.421458,3,0.351126,3,0.228968,...,3,0.667969,3,NaN,3,NaN,3,NaN,3,NaN
2,2020-02-22,0,2,0.405564,0,0.256603,0,0.368448,0,0.258610,...,3,0.792969,0,NaN,2,NaN,0,NaN,0,NaN
3,2020-03-28,1,2,0.361563,1,0.353343,1,0.313902,3,0.281819,...,1,0.695312,1,NaN,1,NaN,1,NaN,1,NaN
4,2020-05-09,1,1,0.369208,2,0.521405,2,0.396920,2,0.264420,...,2,0.585938,1,NaN,1,NaN,1,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,2026-04-09,2,2,0.436148,2,0.477492,2,0.393201,2,0.161335,...,2,0.730469,0,NaN,0,NaN,2,NaN,2,NaN
63,2026-05-14,3,0,0.361564,2,0.256860,0,0.292179,3,0.260099,...,2,0.648438,3,NaN,3,NaN,3,NaN,3,NaN
64,2026-06-18,3,3,0.321707,3,0.407432,3,0.378150,3,0.126798,...,3,0.585938,0,NaN,0,NaN,3,NaN,1,NaN
65,2026-07-23,2,3,0.329653,2,0.369093,2,0.288379,3,0.201083,...,2,0.800781,3,NaN,3,NaN,3,NaN,3,NaN


In [23]:
prediction_cols = [
    "Date",
    "True answer",
    *[f"{model} prediction" for model in model_files]
]

display_df = comparison_df[prediction_cols].copy()

# Add accuracy row
accuracy_row = {"Date": "Test Accuracy", "True answer": ""}

for model in model_files:
    col = f"{model} prediction"
    accuracy = (comparison_df[col] == comparison_df["True answer"]).mean()
    accuracy_row[col] = f"{accuracy:.2%}"

display_df = pd.concat(
    [
        display_df,
        pd.DataFrame([accuracy_row])
    ],
    ignore_index=True
)


def colour_predictions(row):
    styles = []

    for col in display_df.columns:
        if col in ["Date", "True answer"] or row["Date"] == "Test Accuracy":
            styles.append("")
        else:
            correct = row[col] == row["True answer"]

            if correct:
                styles.append(
                    "background-color: #1B5E20; color: white; font-weight: bold;"
                )
            else:
                styles.append(
                    "background-color: #8B1E1E; color: white; font-weight: bold;"
                )

    return styles


styled_df = (
    display_df.style
    .apply(colour_predictions, axis=1)
    .set_properties(**{"text-align": "center"})
    .set_properties(
        subset=pd.IndexSlice[display_df.index[-1], :],
        **{"font-weight": "bold"}
    )
    .hide(axis="index")
)

styled_df

Date,True answer,TF-IDF + LR prediction,MiniLM + LR prediction,MiniLM + MLP prediction,DistilBERT Headline prediction,Distilbert Concat prediction,Distilbert Shared prediction,Llama QLoRA prediction,Claude Haiku 4.5 prediction,GPT-5 Nano prediction,Claude Fable 5.1 prediction,GPT-6 Astra prediction
2019-11-23,0,3,0,0,0,0,0,1,1,1,0,0
2020-01-18,3,3,3,3,3,0,3,3,3,3,3,3
2020-02-22,0,2,0,0,0,0,0,3,0,2,0,0
2020-03-28,1,2,1,1,3,0,3,1,1,1,1,1
2020-05-09,1,1,2,2,2,0,2,2,1,1,1,1
2020-06-13,0,1,1,1,1,0,1,1,1,2,3,3
2020-07-25,1,1,2,2,1,0,1,2,0,1,1,1
2020-08-29,2,3,2,2,0,3,0,0,2,1,2,2
2020-10-03,3,3,2,0,0,0,3,1,1,3,3,3
2020-11-05,1,3,2,2,3,0,3,1,3,3,1,1


In [20]:
distilbert_check = pd.read_csv("../data/predictions_distilbert_headline.csv")

true_answers = np.array([quiz["answer"] for quiz in test_set])

csv_accuracy = np.mean(
    distilbert_check["prediction"].to_numpy() == true_answers
)

print(f"CSV accuracy: {csv_accuracy:.4%}")
print(f"Rows: {len(distilbert_check)}")

CSV accuracy: 47.7612%
Rows: 67
